# 7.1 碰并与破碎过程

**学习目标**
- 理解雨滴碰并和破碎过程的物理机制
- 观察DSD变化对极化变量的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 7, pp. 661-700

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def coalescence_effect(dsd_before, collision_efficiency=0.5):
    """模拟碰并过程对滴谱的影响"""
    # 简化：碰并使大滴增大，小滴减少
    D = np.linspace(0.1, 8, 100)
    # 权重：大滴更易碰并
    weight = (D / D.max())**2 * collision_efficiency
    dsd_after = dsd_before * (1 + weight)
    # 守恒：总数减少
    dsd_after = dsd_after * (1 - collision_efficiency * 0.3)
    return D, dsd_after

def breakup_effect(dsd_before, breakup_rate=0.2):
    """模拟破碎过程"""
    D = np.linspace(0.1, 8, 100)
    # 简化：大滴破碎产生多个小滴
    large_drop_mask = D > 2.0
    dsd_after = dsd_before.copy()
    # 大滴减少
    dsd_after[large_drop_mask] *= (1 - breakup_rate)
    # 小滴增加（来自大滴破碎）
    small_increase = breakup_rate * 0.5 * dsd_before[large_drop_mask].sum() / (~large_drop_mask).sum()
    dsd_after[~large_drop_mask] += small_increase
    return D, dsd_after

def plot_microphysical_processes():
    """可视化碰并破碎过程"""
    D = np.linspace(0.1, 8, 100)
    N0 = 8000
    Lambda = 4.1 * 10**(-0.21)
    dsd_initial = N0 * np.exp(-Lambda * D)
    
    # 碰并和破碎后的滴谱
    D_coal, dsd_coal = coalescence_effect(dsd_initial, 0.4)
    D_break, dsd_break = breakup_effect(dsd_initial, 0.3)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 滴谱变化
    ax1 = axes[0, 0]
    ax1.semilogy(D, dsd_initial, 'b-', linewidth=2, label='初始滴谱')
    ax1.semilogy(D_coal, dsd_coal, 'g--', linewidth=2, label='碰并后')
    ax1.semilogy(D_break, dsd_break, 'r:', linewidth=2, label='破碎后')
    ax1.set_xlabel('直径 D (mm)', fontsize=11)
    ax1.set_ylabel('N(D) (mm⁻¹ m⁻³)', fontsize=11)
    ax1.set_title('碰并破碎对滴谱的影响', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, 8)
    ax1.set_ylim(1, 1e5)
    
    # 对Z和ZDR的估计影响
    ax2 = axes[0, 1]
    D6_initial = np.sum(dsd_initial * D**6) * (D[1]-D[0])
    D6_coal = np.sum(dsd_coal * D_coal**6) * (D_coal[1]-D_coal[0])
    D6_break = np.sum(dsd_break * D_break**6) * (D_break[1]-D_break[0])
    
    processes = ['初始', '碰并后', '破碎后']
    Z_values = [10*np.log10(D6_initial+1e-10)+30, 
                10*np.log10(D6_coal+1e-10)+30,
                10*np.log10(D6_break+1e-10)+30]
    colors = ['blue', 'green', 'red']
    bars = ax2.bar(processes, Z_values, color=colors, alpha=0.7)
    ax2.set_ylabel('估计 Z (dBZ)', fontsize=11)
    ax2.set_title('反射率变化', fontsize=12)
    for bar, val in zip(bars, Z_values):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}', ha='center')
    ax2.grid(True, alpha=0.3)
    
    # 物理过程示意
    ax3 = axes[1, 0]
    time = np.linspace(0, 30, 100)
    Z_t = 30 + 5*np.sin(2*np.pi*time/30) + np.random.randn(100)*1
    ZDR_t = 2 + 1*np.cos(2*np.pi*time/30) + np.random.randn(100)*0.2
    ax3.plot(time, Z_t, 'b-', linewidth=1.5, label='Z')
    ax3_twin = ax3.twinx()
    ax3_twin.plot(time, ZDR_t, 'g-', linewidth=1.5, label='ZDR')
    ax3.set_xlabel('时间 (分钟)', fontsize=11)
    ax3.set_ylabel('Z (dBZ)', color='blue', fontsize=11)
    ax3_twin.set_ylabel('ZDR (dB)', color='green', fontsize=11)
    ax3.set_title('碰并破碎过程中Z和ZDR的变化', fontsize=12)
    ax3.grid(True, alpha=0.3)
    
    # ZDR变化解释
    ax4 = axes[1, 1]
    ax4.axis('off')
    explanation = """
    碰并破碎过程的极化特征：
    
    碰并过程：
    - 大滴增多，小滴减少
    - Z 增加，ZDR 增加
    - ρhv 略微降低
    
    破碎过程：
    - 大滴减少，小滴增多
    - Z 变化较小
    - ZDR 降低
    
    实际降水中碰并和破碎同时发生，
    导致DSD达到准平衡态。
    """
    ax4.text(0.1, 0.5, explanation, fontsize=11, verticalalignment='center',
             family='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow'))
    ax4.set_title('微物理过程解释', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 碰并破碎的极化指纹 ===")
    print("碰并: Z↑, ZDR↑, ρhv↓")
    print("破碎: Z变化小, ZDR↓, ρhv↓")

In [ ]:
plot_microphysical_processes()

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 7, Springer.